# Preprocessing

In [1]:
import sys; sys.path.append("..")
import numpy as np
import pandas as pd
from pathlib import Path
from src.aggregate import aggregate

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")

In [2]:
cc = pd.read_csv(RAW / "credit_card_balance.csv")
cc["util"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"]
cc["min_pay_ratio"] = cc["AMT_PAYMENT_CURRENT"] / cc["AMT_INST_MIN_REGULARITY"]
cc["cash_share"] = cc["AMT_DRAWINGS_ATM_CURRENT"] / cc["AMT_DRAWINGS_CURRENT"]
cc["is_dpd"] = (cc["SK_DPD"] > 0).astype("int8")
cc = cc.replace([np.inf, -np.inf], np.nan)
cc.shape

(3840312, 27)

# Feature Engineering

## Iteration 1

In [3]:
c = aggregate(cc, "SK_ID_CURR", "cc", cat_cols=["NAME_CONTRACT_STATUS"], ignore=["SK_ID_PREV"])
c["cc_cash_card_ratio"] = c["cc_AMT_DRAWINGS_ATM_CURRENT_sum"] / c["cc_AMT_DRAWINGS_CURRENT_sum"]
c = c.replace([np.inf, -np.inf], np.nan)

In [4]:
cc = cc.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"]).reset_index(drop=True)
cc["blk"] = (cc["SK_ID_PREV"].ne(cc["SK_ID_PREV"].shift()) | cc["is_dpd"].ne(cc["is_dpd"].shift())).cumsum()
runs = cc[cc["is_dpd"] == 1].groupby(["SK_ID_PREV", "blk"]).size()
loan = pd.DataFrame({
    "curr": cc.groupby("SK_ID_PREV")["SK_ID_CURR"].first(),
    "streak": runs.groupby("SK_ID_PREV").max(),
    "currently_dpd": cc.groupby("SK_ID_PREV")["is_dpd"].last(),
})
loan["streak"] = loan["streak"].fillna(0)
gc = loan.groupby("curr")
c = c.join(pd.DataFrame({
    "cc_n_cards": gc.size(),
    "cc_max_dpd_streak": gc["streak"].max(),
    "cc_any_currently_dpd": gc["currently_dpd"].max(),
}))
c.shape

(103558, 149)

In [5]:
parts = []
for m in [6, 12, 24, 60]:
    w = cc[cc["MONTHS_BALANCE"] >= -m].groupby("SK_ID_CURR")
    parts += [w["util"].max().rename(f"cc_util_max_{m}"),
              w["util"].mean().rename(f"cc_util_mean_{m}"),
              w["is_dpd"].mean().rename(f"cc_dpd_share_{m}"),
              w["SK_DPD"].max().rename(f"cc_dpd_max_{m}"),
              w["min_pay_ratio"].mean().rename(f"cc_min_pay_ratio_{m}")]
windows = pd.concat(parts, axis=1)

frac = {}
for s, l in [(6, 12), (12, 24), (24, 60)]:
    frac[f"cc_util_frac_{s}_{l}"] = windows[f"cc_util_mean_{s}"] / windows[f"cc_util_mean_{l}"]
frac = pd.DataFrame(frac).replace([np.inf, -np.inf], np.nan)

du = cc.dropna(subset=["util"]); gu = du.groupby("SK_ID_CURR")
dxu = du["MONTHS_BALANCE"] - gu["MONTHS_BALANCE"].transform("mean")
dyu = du["util"] - gu["util"].transform("mean")
util_trend = (dxu * dyu).groupby(du["SK_ID_CURR"]).sum() / (dxu * dxu).groupby(du["SK_ID_CURR"]).sum()
g = cc.groupby("SK_ID_CURR")
dxd = cc["MONTHS_BALANCE"] - g["MONTHS_BALANCE"].transform("mean")
dyd = cc["SK_DPD"] - g["SK_DPD"].transform("mean")
dpd_trend = (dxd * dyd).groupby(cc["SK_ID_CURR"]).sum() / (dxd * dxd).groupby(cc["SK_ID_CURR"]).sum()

recent = cc.sort_values("MONTHS_BALANCE").groupby("SK_ID_CURR").tail(1)[["SK_ID_CURR", "SK_ID_PREV"]]
ll = cc.merge(recent, on=["SK_ID_CURR", "SK_ID_PREV"]).groupby("SK_ID_CURR")
lastloan = pd.DataFrame({"cc_last_card_util_mean": ll["util"].mean(),
                         "cc_last_card_util_max": ll["util"].max(),
                         "cc_last_card_dpd_share": ll["is_dpd"].mean()})

c = c.join(windows).join(frac).join(lastloan)
c["cc_util_trend"] = util_trend.replace([np.inf, -np.inf], np.nan)
c["cc_dpd_trend"] = dpd_trend.replace([np.inf, -np.inf], np.nan)
c.shape

(103558, 177)

## Iteration 2

In [6]:
for status, tag in [("Active", "active"), ("Completed", "done")]:
    sub = cc[cc["NAME_CONTRACT_STATUS"] == status]
    a = aggregate(sub, "SK_ID_CURR", f"cc_{tag}", ignore=["SK_ID_PREV", "blk"])
    c = c.join(a)
c = c.replace([np.inf, -np.inf], np.nan)
c.shape

(103558, 427)

# Save

In [7]:
c.reset_index().to_pickle(INTERIM / "credit_card_balance.pkl")
c.shape

(103558, 427)